# Convolutional Neural Network

In [1]:
import torch, torchvision, re, os
import numpy as np
from PIL import Image
from collect_data import collect_data, train_model

DATA_DIR = '../data/images/climbs'

# Transform images to batch_size x channels x rows x columns
preprocess = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
])

train_loader, val_loader, test_loader = collect_data(DATA_DIR, preprocess)

In [2]:
# (for now) simple 2 layer Convolution with average pooling and 1 fully connected layer
class CNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(3, 64, 3, padding='same')
        self.pool1 = torch.nn.AvgPool2d(2, 2)
        self.conv2 = torch.nn.Conv2d(64, 128, 3, padding='same')
        self.pool2 = torch.nn.AvgPool2d(2, 2)
        self.conv3 = torch.nn.Conv2d(128, 128, 3, padding='same')
        self.pool3 = torch.nn.AvgPool2d(2, 2)
        self.fc1 = torch.nn.Linear(100225, 4096) # 128 * 27 * 29 input features + 1 angle
        self.fc2 = torch.nn.Linear(4096, 23)
        self.flatten = torch.nn.Flatten()
        self.relu = torch.nn.ReLU()
    
    def forward(self, X, angles):
        c1 = self.conv1(X)
        r1 = self.relu(c1)
        p1 = self.pool1(r1)
        
        c2 = self.conv2(p1)
        r2 = self.relu(c2)
        p2 = self.pool2(r2)
        
        c3 = self.conv3(p2)
        r3 = self.relu(c3)
        p3 = self.pool3(r3)
        
        f = self.flatten(p3)
        f = torch.cat([f, angles], dim=1)
        z1 = self.fc1(f)
        a1 = self.relu(z1)
        
        y = self.fc2(a1)
        return y


In [3]:
device = torch.device('cpu') 
if torch.cuda.is_available():
    device = torch.device('cuda') # rose laptops and the servers
elif torch.backends.mps.is_available():
    device = torch.device('mps') # mac
print('Using device:', device)

Using device: cuda


In [4]:
model = CNN().to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params}')

Total trainable parameters: 410843159


In [5]:
# Hyperparameters
epsilon = 1E-4

# Use CCE loss
criterion = torch.nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=epsilon)

train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=10, checkpoint_path='kilter_cnn.pth')

Epoch [0/10], Train Loss: 3.12576, Val Loss: 2.67705 *, Train Acc: 0.11429, Validation Acc: 0.07143
Epoch [1/10], Train Loss: 2.67602, Val Loss: 2.86118, Train Acc: 0.21429, Validation Acc: 0.07143
Epoch [2/10], Train Loss: 2.60737, Val Loss: 2.88095, Train Acc: 0.27143, Validation Acc: 0.07143
Epoch [3/10], Train Loss: 2.57821, Val Loss: 2.88619, Train Acc: 0.22857, Validation Acc: 0.05714
Epoch [4/10], Train Loss: 2.49093, Val Loss: 2.93670, Train Acc: 0.24286, Validation Acc: 0.00000
Epoch [5/10], Train Loss: 2.55494, Val Loss: 2.96370, Train Acc: 0.08571, Validation Acc: 0.00000
Epoch [6/10], Train Loss: 2.66108, Val Loss: 2.90395, Train Acc: 0.08571, Validation Acc: 0.00000
Epoch [7/10], Train Loss: 2.68831, Val Loss: 2.77799, Train Acc: 0.07143, Validation Acc: 0.00000
Epoch [8/10], Train Loss: 2.51389, Val Loss: 2.65472 *, Train Acc: 0.08571, Validation Acc: 0.05714
Epoch [9/10], Train Loss: 2.44005, Val Loss: 2.55426 *, Train Acc: 0.34286, Validation Acc: 0.07143
Average time p